In [ ]:
import sys; sys.path.append('..')
import MeshFEM, elastic_sheet, tensors
import numpy as np
np.set_printoptions(edgeitems=12,linewidth=280)

In [ ]:
import fd_validation

In [ ]:
class FDWrap:
    def __init__(self):
        self.pbe = elastic_sheet.PlateBendingElement(1)
        self.X = np.random.normal(size=(3,3))
        self.x = self.X + 0.1 * np.random.normal(size=(3,3))
        self.gamma = 0.01 * np.random.normal(size=3)
        self.C = tensors.ElasticityTensor2D(2000, 0.3)
        
    def numVars(self): return 12
    def getVars(self): return np.concatenate((self.x.ravel(), self.gamma))
    def setVars(self, v):
        self.x = v[0:9].reshape(3, 3)
        self.gamma = v[9:]
    def energy(self):
        return self.pbe.energy(self.C, self.X, self.x, self.gamma)
    def gradient(self):
        return self.pbe.gradient(self.C, self.X, self.x, self.gamma)
    def hessian(self):
        return self.pbe.hessian(self.C, self.X, self.x, self.gamma)

In [ ]:
fdwrap = FDWrap()

In [ ]:
fd_validation.gradConvergencePlot(fdwrap)

In [ ]:
fd_validation.hessConvergencePlot(fdwrap, perturb=fd_validation.basisDirection(fdwrap, 10))

In [ ]:
fdwrap.hessian()

In [ ]:
eps = 1e-6
fd_hess = np.zeros_like(fdwrap.hessian())
x = fdwrap.getVars().copy()
for i in range(fd_hess.shape[1]):
    d = fd_validation.basisDirection(fdwrap, i)
    fdwrap.setVars(x + eps * d)
    g_plus = fdwrap.gradient()
    fdwrap.setVars(x - eps * d)
    g_minus = fdwrap.gradient()
    fdwrap.setVars(x)
    fd_hess[:, i] = (g_plus - g_minus) / (2 * eps)
fd_hess